# Phase 5: Parametric Studies and Publication-Quality Results

This notebook performs a multi-dimensional CCE vs dose-rate sweep across epitaxial layer thickness, bulk doping concentration, and bias voltage for the FLASH SiC paper.

**Parameter space:** 4 epi thicknesses x 5 doping levels x 3 bias voltages = 60 conditions, each swept over 6 dose rates (20-230 Gy/s).

**Scientific goal:** Determine whether any combination of detector design parameters leads to measurable CCE degradation under FLASH dose rates in 4H-SiC.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebook creation
import matplotlib.pyplot as plt
from etna.core.flash_recombination import (parametric_cce_sweep, save_parametric_results,
                                      load_parametric_results, cce_vs_dose_rate)
from etna.core.devsim_lock import quiet_devsim
from etna.core.plotting import (plot_parametric_epi, plot_parametric_doping, plot_parametric_bias,
                          plot_cce_vs_dose_rate, save_figure)

print('All imports successful')

## Configuration

Modify these parameters to customize the parametric sweep. The full sweep (60 conditions x 6 dose
rates) takes about 80 s on the current solver stack (measured 2026-10-18, Apple M4), so it runs
whenever the cache is missing or incomplete; set `RECOMPUTE = True` to force it. The cache is
written next to the figures saved by `save_figure` (`$ETNA_FIGURE_DIR`, default `./figures`
of the working directory: `notebooks/figures/` when run from `notebooks/`, gitignored), never
into the tracked `figures/`.

In [ ]:
DOSE_RATES = np.array([20, 50, 100, 150, 200, 230], dtype=float)
EPI_THICKNESSES = [5e-4, 10e-4, 15e-4, 20e-4]
N_D_BULK_VALUES = [5e13, 8.5e13, 1e14, 2e14, 5e14]
BIAS_VOLTAGES = [-10.0, -30.0, -50.0]
E_MEV = 62
RECOMPUTE = False  # True forces the sweep even with a complete cache
FIGURE_DIR = os.environ.get("ETNA_FIGURE_DIR", "figures")
RESULTS_FILE = os.path.join(FIGURE_DIR, "parametric_results.json")

## Parametric Sweep

The sweep explores all combinations of the parameters above:
- **Epi thickness:** 5, 10, 15, 20 um
- **Bulk doping:** 5e13, 8.5e13, 1e14, 2e14, 5e14 cm$^{-3}$
- **Bias voltage:** -10, -30, -50 V

Total: 4 x 5 x 3 = 60 parameter combinations, each swept over 6 dose rates.

In [ ]:
expected_keys = [
    (epi, nd, vb)
    for epi in EPI_THICKNESSES
    for nd in N_D_BULK_VALUES
    for vb in BIAS_VOLTAGES
]
results = load_parametric_results(RESULTS_FILE) if os.path.exists(RESULTS_FILE) else {}
found = sum(1 for k in expected_keys if k in results)
if RECOMPUTE or found < len(expected_keys):
    print(f"Cache has {found}/{len(expected_keys)} conditions: running the full sweep (~80 s)...")
    with quiet_devsim():  # devsim prints ~65 lines per solve otherwise
        results = parametric_cce_sweep(DOSE_RATES, EPI_THICKNESSES, N_D_BULK_VALUES, BIAS_VOLTAGES, E_MEV)
    os.makedirs(FIGURE_DIR, exist_ok=True)
    save_parametric_results(results, RESULTS_FILE)
    found = sum(1 for k in expected_keys if k in results and results[k] is not None)
    print(f"Sweep complete: {found}/{len(expected_keys)} conditions saved to {RESULTS_FILE}")
else:
    print(f"Loaded {len(results)} conditions from {RESULTS_FILE} ({found}/{len(expected_keys)} expected)")

## Results: Epitaxial Thickness Dependence

CCE vs dose rate for varying epitaxial layer thickness at fixed reference doping ($N_D$ = 8.50e13 cm$^{-3}$). One panel per bias voltage.

In [ ]:
fig = plot_parametric_epi(results, EPI_THICKNESSES, N_D_bulk_ref=8.50e13)
save_figure(fig, "flash_parametric_epi")
plt.show()

## Results: Doping Concentration Dependence

CCE vs dose rate for varying bulk doping concentration at fixed reference epi thickness (10 um). One panel per bias voltage.

In [ ]:
fig = plot_parametric_doping(results, N_D_BULK_VALUES, epi_ref_cm=10e-4)
save_figure(fig, "flash_parametric_doping")
plt.show()

## Results: Bias Voltage Dependence

CCE vs dose rate for varying bias voltage at fixed reference epi thickness (10 um) and doping ($N_D$ = 8.50e13 cm$^{-3}$).

In [ ]:
fig = plot_parametric_bias(results, BIAS_VOLTAGES, epi_ref_cm=10e-4, N_D_bulk_ref=8.50e13)
save_figure(fig, "flash_parametric_bias")
plt.show()

## Summary and Key Findings

This parametric study explores how detector design parameters affect charge collection under FLASH dose rates:

1. **Number of parameter combinations:** 60 (4 epi x 5 doping x 3 bias), each swept over 6 dose rates from 20 to 230 Gy/s.

2. **CCE degradation:** The analysis below checks whether any condition produces CCE below 0.99, indicating meaningful Auger-driven degradation.

3. **Implications for SiC dosimeters under FLASH irradiation:** If CCE remains near unity across all conditions, 4H-SiC detectors are robust dose-rate-independent dosimeters regardless of design parameter choice within the explored range.

---
*Phase 5 of the 4H-SiC TCAD simulation project*

In [ ]:
total = len(results)
failed = sum(1 for v in results.values() if v is None)
print(f"Parameter combinations: {total}")
print(f"Failed simulations: {failed}")
print(f"Successful: {total - failed}")
# Check if any CCE < 0.99 (indicating meaningful degradation)
for key, val in results.items():
    if val is not None and np.any(np.array(val["cce_values"]) < 0.99):
        epi, nd, vb = key
        print(f"  CCE degradation at epi={epi*1e4:.0f}um, N_D={nd:.1e}, V={vb}V: min CCE={min(val['cce_values']):.4f}")